# RagForge Agent Exploration Playground

This notebook demonstrates direct interaction with the LangGraph-based `RagForgeAgent` class.

It covers:
1. **Tool Discovery**: Exploring the MCP (Model Context Protocol) tools dynamically discovered from the Qdrant and OpenProject stdio servers.
2. **Read-Only Agent Flow**: Triggering a question that queries Qdrant and showing it executes to completion without pausing.
3. **Write Agent Flow & HITL Interrupt**: Requesting a database write operation and demonstrating how the graph pauses at the `hitl_pause` node.
4. **HITL Resume & Execution**: Resuming the graph with approval, executing the tool, and fetching the final answer.
5. **Chat History Memory Ingestion**: Manually indexing a chat turn into the long-term semantic Qdrant memory collection.

In [1]:
import sys
from pathlib import Path

# Locate and append project root so src.ragforge is importable
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

project_root = get_project_root().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root: {project_root}")

Project root: /Users/sunnyraj/code_files/git_repos/RagForge


In [2]:
import uuid
from src.ragforge.agents.rag import RagForgeAgent
from src.ragforge.config import DEFAULT_LLM_MODEL
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

session_id = f"playground-session-{uuid.uuid4()}"
print(f"Using default LLM model: {DEFAULT_LLM_MODEL}")
print(f"Session ID: {session_id}")

# Initialize and compile the LangGraph-based agent
agent = RagForgeAgent(llm_model=DEFAULT_LLM_MODEL, session_id=session_id)
await agent.initialise()
print("\nAgent initialized and LangGraph compiled successfully!")

Using default LLM model: gemma4:e4b
Session ID: playground-session-30b802e5-1864-4f18-b504-6e262e594d44

Agent initialized and LangGraph compiled successfully!


## 1. Inspect Discovered MCP Tools

The agent dynamically binds tools exposed by the Qdrant and OpenProject MCP servers. Let's list the discovered tools and see which ones are categorized as write operations.

In [3]:
print("Discovered MCP Tools & Mapping:")
tool_map = agent.get_tool_server_map()
for tool_name, server in tool_map.items():
    print(f" - {tool_name:<25} -> Server: {server.upper()}")

print("\nWrite/Mutating tools (triggers Human-in-the-Loop validation):")
for wt in agent.get_write_tools():
    print(f" - {wt}")

Discovered MCP Tools & Mapping:
 - create_collection         -> Server: QDRANT
 - upsert_documents          -> Server: QDRANT
 - search_documents          -> Server: QDRANT
 - search_chat_history       -> Server: QDRANT
 - get_project_list          -> Server: OPENPROJECT
 - get_project_tasks         -> Server: OPENPROJECT
 - create_project_task       -> Server: OPENPROJECT
 - update_task_status        -> Server: OPENPROJECT
 - add_task_comment          -> Server: OPENPROJECT
 - ingest_file_or_directory  -> Server: OPENPROJECT

Write/Mutating tools (triggers Human-in-the-Loop validation):
 - upsert_documents
 - ingest_file_or_directory
 - update_task_status
 - add_task_comment
 - create_project_task


## 2. Run a Read-Only Query (Auto-executes)

When we submit a read-only query (like searching the knowledge base), the graph will auto-route through `execute_tools` and output the final response without hitting any pause.

In [ ]:
thread_config = {"configurable": {"thread_id": session_id}}

input_state = {
    "messages": [HumanMessage(content="Hello! Search the knowledge base for any documents containing 'test'.")],
    "session_id": session_id,
    "pending_tool_call": None,
    "hitl_approved": None,
}

print("Running read-only query...")
async for event in agent.graph.astream(
    input_state,
    config=thread_config,
    stream_mode="values",
):
    last_msg = event["messages"][-1]
    role = type(last_msg).__name__
    print(f"\n[Node Update] Message type: {role}")
    if last_msg.content:
        print(f"Content: {last_msg.content}")
    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        print(f"Requested Tool Calls: {last_msg.tool_calls}")

snapshot = agent.graph.get_state(thread_config)
print("\nExecution complete!")
print(f"Next node to execute: {snapshot.next}")

## 3. Run a Write Query (Triggers HITL Pause)

If we request a write operation, the agent will select a write-gate tool (like `upsert_documents`). The `check_hitl` node will catch this and route the execution to the `hitl_pause` node where it halts before executing.

In [6]:
write_session_id = f"playground-write-session-{uuid.uuid4()}"
thread_config_write = {"configurable": {"thread_id": write_session_id}}

input_state_write = {
    "messages": [HumanMessage(content="Index the document 'LangGraph agent architecture is premium' into the database.")],
    "session_id": write_session_id,
    "pending_tool_call": None,
    "hitl_approved": None,
}

print("Running write query...")
async for event in agent.graph.astream(
    input_state_write,
    config=thread_config_write,
    stream_mode="values",
):
    pass

# Check checkpoint values and inspect current pause point
snapshot = agent.graph.get_state(thread_config_write)
print("\n--- Paused State Info ---")
print(f"Next node to execute : {snapshot.next}")
print(f"Pending Tool Call    : {snapshot.values.get('pending_tool_call')}")
print(f"HITL Approved Status : {snapshot.values.get('hitl_approved')}")

Running write query...

--- Paused State Info ---
Next node to execute : ('hitl_pause',)
Pending Tool Call    : {'id': '6b7213d9-e135-4efb-9f51-655a0ec07df4', 'name': 'upsert_documents', 'args': {'documents': [{'text': 'LangGraph agent architecture is premium'}], 'session_id': 'playground-write-session-7405d50f-a506-4357-839c-f4a063ad169c'}}
HITL Approved Status : None


## 4. Approve and Resume Execution

Now we simulate user approval. We update the state checkpointer with approval metadata, then resume the graph execution by passing `None` as the new input.

In [7]:
# Update state variables to grant approval
state_update = {"hitl_approved": True, "pending_tool_call": None}
agent.graph.update_state(thread_config_write, state_update, as_node="check_hitl")

print("Resuming graph with approval...")
async for event in agent.graph.astream(
    None, # Pass None to resume from checkpoint
    config=thread_config_write,
    stream_mode="values",
):
    last_msg = event["messages"][-1]
    role = type(last_msg).__name__
    print(f"\n[Node Update] Message type: {role}")
    if last_msg.content:
        print(f"Content: {last_msg.content}")

final_snapshot = agent.graph.get_state(thread_config_write)
print("\nExecution complete!")
print(f"Next node to execute: {final_snapshot.next}")

Resuming graph with approval...

[Node Update] Message type: AIMessage

[Node Update] Message type: ToolMessage
Content: Successfully upserted 1 documents into collection 'ragforge-collection'.

[Node Update] Message type: AIMessage
Content: The document 'LangGraph agent architecture is premium' has been successfully indexed into the database.

Is there anything else I can help you with, such as querying this new information, managing project tasks, or ingesting more documents?

Execution complete!
Next node to execute: ()


## 5. Long-Term Memory Recall Indexing

Once the assistant answers, we can index the completed turn into Qdrant so that the agent can remember this context when requested later.

In [8]:
from src.ragforge.agents.rag import index_chat_turn

user_msg = "Index the document 'LangGraph agent architecture is premium' into the database."
assistant_msg = "The document 'LangGraph agent architecture is premium' has been successfully indexed into the database."

print(f"Indexing completed turn into chat memory store for Session: {write_session_id}...")
await index_chat_turn(write_session_id, user_msg, assistant_msg)
print("Turn indexed successfully! The context is now retrievable in subsequent chats.")

Indexing completed turn into chat memory store for Session: playground-write-session-7405d50f-a506-4357-839c-f4a063ad169c...
Turn indexed successfully! The context is now retrievable in subsequent chats.
